# CodeCourt — Colab Training Notebook

This notebook runs the full CodeCourt training pipeline in Google Colab.

## Setup

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/ayushoncode/CodeCourt.git
%cd CodeCourt
!pip install -r requirements.txt

## Configuration

In [ ]:
import argparse
args = argparse.Namespace(
    episodes=200,
    model="Qwen/Qwen2.5-0.5B-Instruct",
    use_wandb=True,
    save_every=50,
    output_dir="./outputs",
    mock=False
)

## Load Model (Unsloth 4-bit)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=args.model,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"✓ Loaded {args.model} with Unsloth (4-bit quantized)")

## Initialize Agents & Environment

In [ ]:
import sys
import os
import json

sys.path.insert(0, '.')

from env.codecourt_env import CodeCourtEnv
from agents.setter import SetterAgent
from agents.solver import SolverAgent

# Initialize
setter = SetterAgent(model=model, tokenizer=tokenizer)
solver = SolverAgent(model=model, tokenizer=tokenizer)
env = CodeCourtEnv(difficulty_progression=True)

print("✓ Agents and environment initialized")

## Training Loop

In [ ]:
import wandb

if args.use_wandb:
    wandb.init(project="codecourt", config=vars(args))

os.makedirs(args.output_dir, exist_ok=True)
history = []

print(f"\n{'='*60}")
print(f"  CodeCourt Training — {args.episodes} episodes")
print(f"{'='*60}\n")

for ep in range(args.episodes):
    obs = env.reset()
    full_problem = env._current_state.problem

    # Setter generates solution
    setter_code = setter.generate_solution(full_problem)

    # Solver attempts solution
    solver_code = solver.solve(full_problem)

    # Step environment
    setter_info, solver_info, done, info = env.step(setter_code, solver_code)

    setter_r = setter_info["reward"]
    solver_r = solver_info["reward"]
    outcome = info["outcome"]

    # Log
    record = {
        "episode": ep,
        "archetype": obs["archetype"],
        "task_id": obs["task_id"],
        "difficulty": obs["difficulty"],
        "setter_reward": setter_r,
        "solver_reward": solver_r,
        "outcome": outcome,
        "setter_elo": info["elo"]["setter_elo"],
        "solver_elo": info["elo"]["solver_elo"],
        "solver_pass_rate": info["solver_pass_rate"],
    }
    history.append(record)

    if args.use_wandb:
        wandb.log(record)

    # Console progress
    if ep % 10 == 0 or ep < 5:
        print(
            f"Ep {ep:4d} | {obs['archetype']:5s} D{obs['difficulty']} | "
            f"outcome={outcome:12s} | "
            f"S_R={setter_r:+6.1f} V_R={solver_r:+6.1f} | "
            f"Elo: Set={info['elo']['setter_elo']:.0f} "
            f"Sol={info['elo']['solver_elo']:.0f}"
        )

    # Save checkpoint
    if (ep + 1) % args.save_every == 0:
        ckpt_path = os.path.join(args.output_dir, f"history_ep{ep+1}.json")
        with open(ckpt_path, 'w') as f:
            json.dump(history, f, indent=2)
        print(f"  → Checkpoint saved: {ckpt_path}")

        model.save_pretrained(
            os.path.join(args.output_dir, f"model_ep{ep+1}")
        )

## Save Final Results

In [ ]:
# Final save
final_path = os.path.join(args.output_dir, "training_history.json")
with open(final_path, 'w') as f:
    json.dump(history, f, indent=2)

metrics = env.get_metrics()
print(f"\n{'='*60}")
print("  Training Complete")
print(f"{'='*60}")
print(json.dumps(metrics, indent=2))
print(f"\nHistory saved to: {final_path}")

if args.use_wandb:
    wandb.finish()

## Generate Evaluation Plots

In [ ]:
# Run evaluation script
!python scripts/evaluate.py \
    --baseline ./outputs/baseline_results.json \
    --trained ./outputs/training_history.json \
    --output ./outputs/plots/

## Download Results

In [ ]:
from google.colab import files

# Download plots
import shutil
shutil.make_archive("codecourt_outputs", 'zip', "./outputs")
files.download("codecourt_outputs.zip")